# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pankaj1281/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [29]:
import os

REPO_URL = "https://github.com/pankaj1281/flyrank_ml_internship.git"
REPO_PATH = "/content/flyrank_ml_internship"

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL}

print("Repository ready.")

Repository ready.


In [30]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank_ml_internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

Dataset shape: (30000, 44)


# My Rule and Its Reason Codes

## Baseline Rule

The purpose of this baseline rule is to identify pages that are good candidates for content refresh.

The rule assigns a score using only information that would be available before making a refresh decision.

Pages receive a higher score if they:
- have not been updated recently,
- have low CTR despite good search position,
- have high impressions but relatively few clicks.

This rule is designed for decision support and does not use future information or label-derived fields.

## Reason Codes

- STALE_CONTENT
- LOW_CTR
- HIGH_IMPRESSIONS_LOW_CTR

In [31]:
signal1 = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
)

signal1

,n,avg_ctr,avg_impressions
freshness_tier,,,
0-30,20480,0.609021,4199.614062
181+,174,3.693276,1172.448276
31-90,175,0.117543,6506.748571
91-180,9171,0.238367,7486.665140


### Signal Check 1 Verdict

**Verdict:** MIXED

The results show that freshness alone is not a consistent indicator of performance. The 181+ day group has the highest average CTR but contains very few pages, so freshness should be combined with other signals.


In [32]:
signal2 = (
    df.groupby("position_tier")
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean")
      )
)

signal2

,n,avg_ctr
position_tier,,
deep,1319,0.150212
page_1,11814,0.652467
page_3_5,7242,0.222484
striking,7304,0.323239
top_3,2321,1.483611


### Signal Check 2 Verdict

**Verdict:** CONFIRMED

The results show that pages with better search positions generally have higher CTR. This supports using **CTR** and **average position** together as signals in the baseline scoring rule.


# Build the Ranked Queue

Each page receives a baseline score based on predefined rules.

The score is used to prioritize pages for manual review or content refresh.

The notebook exports the ranked results to:

work/outputs/baseline_action_score.csv

In [33]:
df["baseline_score"] = 0
df["reason_code"] = ""

# Rule 1: Stale content
mask = df["days_since_last_update"] >= 180
df.loc[mask, "baseline_score"] += 40
df.loc[mask, "reason_code"] += "STALE_CONTENT;"

# Rule 2: Low CTR but good ranking
mask = (df["avg_position"] <= 10) & (df["ctr"] < 1)
df.loc[mask, "baseline_score"] += 30
df.loc[mask, "reason_code"] += "LOW_CTR;"

# Rule 3: High impressions but low clicks
mask = (df["impressions_90d"] > 1000) & (df["ctr"] < 1)
df.loc[mask, "baseline_score"] += 30
df.loc[mask, "reason_code"] += "HIGH_IMPRESSIONS_LOW_CTR;"

df["action"] = np.select(
    [
        df["baseline_score"] >= 70,
        df["baseline_score"] >= 40
    ],
    [
        "Refresh Immediately",
        "Review"
    ],
    default="Monitor"
)

queue = df.sort_values(
    "baseline_score",
    ascending=False
)

queue.head(20)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
22872,content_e3ff1b093148,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,4758.0,33575.0,...,0.00,20.00,0.0,moderate,page_1,down,-68.5,100,STALE_CONTENT;LOW_CTR;HIGH_IMPRESSIONS_LOW_CTR;,Refresh Immediately
5003,content_17aa56bdad68,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,781.0,5651.0,...,0.00,0.00,0.0,low,page_1,stable,0.0,70,STALE_CONTENT;LOW_CTR;,Refresh Immediately
5653,content_10b9f5f766b4,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,745.0,5866.0,...,0.00,100.00,0.0,low,page_1,down,-88.9,70,STALE_CONTENT;LOW_CTR;,Refresh Immediately
17510,content_b694314765e5,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,789.0,5436.0,...,0.00,100.00,0.0,low,page_1,down,-100.0,70,STALE_CONTENT;LOW_CTR;,Refresh Immediately
14545,content_e2fb3f55bed3,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,927.0,7085.0,...,0.00,100.00,0.0,low,top_3,new,NaN,70,STALE_CONTENT;LOW_CTR;,Refresh Immediately
2519,content_0edf498ae135,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,1034.0,7308.0,...,0.00,0.00,0.0,low,top_3,flat,NaN,70,STALE_CONTENT;LOW_CTR;,Refresh Immediately
22594,content_146814e0dd01,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,1299.0,9533.0,...,0.00,0.00,0.0,low,top_3,flat,NaN,70,STALE_CONTENT;LOW_CTR;,Refresh Immediately
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,5125.0,33705.0,...,0.84,24.11,0.0,excellent,striking,down,-85.6,70,STALE_CONTENT;HIGH_IMPRESSIONS_LOW_CTR;,Refresh Immediately
10836,content_e748f498b262,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,1367.0,13108.0,...,0.00,100.00,0.0,low,top_3,new,NaN,70,STALE_CONTENT;LOW_CTR;,Refresh Immediately
21984,content_02b0d6e30129,client_19581e27de,110.0,0.40,MEDIUM,0.59,keyword article,transactional,NaN,NaN,...,0.00,0.00,0.0,low,page_1,down,-95.6,70,STALE_CONTENT;LOW_CTR;,Refresh Immediately


In [34]:
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv created successfully.")

baseline_action_score.csv created successfully.


# Top-20 Review

For each page, I reviewed:

- Action
- Reason Code
- Confidence
- What would make the recommendation wrong

In [35]:
top20 = queue[
    [
        "content_id",
        "baseline_score",
        "action",
        "reason_code"
    ]
].head(20)

top20

,content_id,baseline_score,action,reason_code
22872,content_e3ff1b093148,100,Refresh Immediately,STALE_CONTENT;LOW_CTR;HIGH_IMPRESSIONS_LOW_CTR;
5003,content_17aa56bdad68,70,Refresh Immediately,STALE_CONTENT;LOW_CTR;
5653,content_10b9f5f766b4,70,Refresh Immediately,STALE_CONTENT;LOW_CTR;
17510,content_b694314765e5,70,Refresh Immediately,STALE_CONTENT;LOW_CTR;
14545,content_e2fb3f55bed3,70,Refresh Immediately,STALE_CONTENT;LOW_CTR;
2519,content_0edf498ae135,70,Refresh Immediately,STALE_CONTENT;LOW_CTR;
22594,content_146814e0dd01,70,Refresh Immediately,STALE_CONTENT;LOW_CTR;
16751,content_cf56e2e2e282,70,Refresh Immediately,STALE_CONTENT;HIGH_IMPRESSIONS_LOW_CTR;
10836,content_e748f498b262,70,Refresh Immediately,STALE_CONTENT;LOW_CTR;
21984,content_02b0d6e30129,70,Refresh Immediately,STALE_CONTENT;LOW_CTR;


# Top-20 Review

### Content ID: content_e3ff1b093148

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR; HIGH_IMPRESSIONS_LOW_CTR

**Confidence:** High

**What would make it wrong?**
The page may be affected by seasonal demand or search intent rather than outdated content.

---

### Content ID: content_17aa56bdad68

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR

**Confidence:** High

**What would make it wrong?**
A temporary drop in CTR or changes in search results could explain the low performance.

---

### Content ID: content_10b9f5f766b4

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR

**Confidence:** High

**What would make it wrong?**
The content may still satisfy user intent despite a lower CTR.

---

### Content ID: content_b694314765e5

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR

**Confidence:** High

**What would make it wrong?**
External factors, such as competitor updates or SERP changes, may have reduced CTR.

---

### Content ID: content_e2fb3f55bed3

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR

**Confidence:** High

**What would make it wrong?**
The page may have stable conversions even if CTR is low.

---

### Content ID: content_0edf498ae135

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR

**Confidence:** High

**What would make it wrong?**
The low CTR may be caused by search intent rather than content quality.

---

### Content ID: content_146814e0dd01

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR

**Confidence:** High

**What would make it wrong?**
CTR could improve naturally if search demand changes.

---

### Content ID: content_cf56e2e2e282

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; HIGH_IMPRESSIONS_LOW_CTR

**Confidence:** High

**What would make it wrong?**
High impressions may be driven by broad search visibility, while low clicks could result from misleading titles or SERP features.

---

### Content ID: content_e748f498b262

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR

**Confidence:** High

**What would make it wrong?**
The page may already meet user needs, reducing click-through improvements after a refresh.

---

### Content ID: content_02b0d6e30129

**Action:** Refresh Immediately

**Reason Code:** STALE_CONTENT; LOW_CTR

**Confidence:** High

**What would make it wrong?**
Performance may be influenced by seasonal trends instead of outdated content.

---

The remaining pages have the same action (**Refresh Immediately**) and the same reasoning (**STALE_CONTENT; LOW_CTR**). They were ranked highly because they satisfied the baseline scoring rules. These pages should be reviewed manually before publishing updates to confirm that the observed performance is not caused by seasonal demand, search intent, or temporary SERP changes.


# Weak Picks

Some pages may be selected because of seasonal traffic patterns rather than content quality.

Pages with low CTR may also be affected by search intent or SERP features instead of needing a refresh.

Manual review is recommended before taking action.

# Leakage Check

No future-window information was used.

No label-derived fields were used.

Only information available before the decision was used in the baseline rule.

In [36]:
queue.tail(20)[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

,content_id,baseline_score,reason_code,action
6371,content_2acabcc66d4d,0,,Monitor
6370,content_180c6a49734c,0,,Monitor
6368,content_8966697301a8,0,,Monitor
29999,content_887020f20b5e,0,,Monitor
19852,content_d1e00a4f4559,0,,Monitor
19850,content_363e0f634994,0,,Monitor
19847,content_d7dbdb3949ce,0,,Monitor
19846,content_152a1702a7c0,0,,Monitor
19845,content_aa7f06d056d0,0,,Monitor
19844,content_e157c0b5d68d,0,,Monitor


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.